# MultiVI Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using MultiVI on simulated dataset.

## Loading

In [ ]:
import omicverse as ov

import gzip
import os
import tempfile
from pathlib import Path

import numpy as np
import pooch
import scanpy as sc
import scvi
import seaborn as sns
import anndata as ad
import torch

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

## MultiVI pipeline

In [ ]:
# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA and ATAC datasets
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_rna = adata_rna.raw.to_adata()
    adata_rna.var['modality'] = 'Gene Expression'
    
    adata_atac = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    sc.pp.filter_genes(adata_atac, min_cells=1)
    adata_atac.var['modality'] = 'Peaks'

    # Identify highly variable genes and peaks
    sc.pp.highly_variable_genes(adata_rna, n_top_genes=3000)
    adata_rna = adata_rna[:, adata_rna.var['highly_variable'] == True]
    sc.pp.highly_variable_genes(adata_atac, n_top_genes=50000)
    adata_atac = adata_atac[:, adata_atac.var['highly_variable'] == True]

    # Concatenate the RNA and ATAC data
    adata_paired = ad.concat([adata_rna, adata_atac], axis=1)
    adata_paired.obs['ground_truth'] = adata_rna.obs['cell_type']
    adata_paired.obsm['spatial'] = adata_rna.obsm['spatial']

    # Organize multiome data
    adata_mvi = scvi.data.organize_multiome_anndatas(adata_paired, adata_rna, adata_atac)

    # Set up the model
    scvi.model.MULTIVI.setup_anndata(adata_mvi, batch_key="modality")
    model = scvi.model.MULTIVI(
        adata_mvi,
        n_genes=len(adata_rna.var_names),
        n_regions=len(adata_atac.var_names),
    )
    model.view_anndata_setup()
    model.train()

    # Get latent representation and perform UMAP
    MULTIVI_LATENT_KEY = "X_multivi"
    adata_mvi.obsm[MULTIVI_LATENT_KEY] = model.get_latent_representation()
    sc.pp.neighbors(adata_mvi, use_rep=MULTIVI_LATENT_KEY)
    sc.tl.umap(adata_mvi, min_dist=0.2)
    sc.pl.umap(adata_mvi, color="modality")

    # Filter paired data
    adata_paired = adata_mvi[adata_mvi.obs['modality'] == 'paired']

    # Perform clustering using the latent representation
    ov.pp.neighbors(adata_paired, n_neighbors=15, n_pcs=adata_paired.obsm['X_multivi'].shape[1],
                    use_rep='X_multivi')
    ov.utils.cluster(adata_paired, use_rep='X_multivi', method='leiden', resolution=0.27)

    # Plot spatial clustering results
    sc.pl.spatial(adata_paired, color=['ground_truth', 'leiden'], spot_size=0.12, wspace=0.4)

    # Save the processed dataset
    output_path = f'{output_dir}/Simulated_Dataset_{i}/multivi_multiomics.h5ad'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    adata_paired.write_h5ad(output_path, compression='gzip')

In [ ]:
!pip list